# KV Cache Analysis - VLM vs Text Token Distribution

Focused analysis of KV-cache quantization behavior:

1. **prod (Algorithm 2) failure rate** vs MSE-only (Algorithm 1)
2. **VLM vs Text bitwidth sensitivity** (score drop at tq-3)
3. **Asymmetric K/V effectiveness** (tq-K4V2 vs tq-3, tq-K4V3 vs tq-3h)

In [ ]:
import sys
import json
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, str(Path('..').resolve()))

from tq_bench.config import load_benchmarks

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110

In [ ]:
RESULTS_DIR = Path('../../results').resolve()
CONFIG_DIR = Path('../configs').resolve()
runs_dir = RESULTS_DIR / 'runs'

rows = []
for p in sorted(runs_dir.glob('*.json')):
    if p.name == 'checkpoint.json':
        continue
    with open(p) as f:
        r = json.load(f)
    if 'runtime_id' not in r or 'benchmark_id' not in r:
        continue
    rows.append({
        'model_id':      r.get('model_id'),
        'runtime_id':   r.get('runtime_id'),
        'benchmark_id': r.get('benchmark_id'),
        'score':        r.get('score'),
        'n_samples':    r.get('n_samples'),
        'n_failed':     r.get('n_failed', 0),
        'status':       r.get('status', 'ok'),
    })
df = pd.DataFrame(rows)
available_models = sorted(df['model_id'].dropna().unique().tolist())
ACTIVE_MODEL_ID = available_models[0] if available_models else None
if ACTIVE_MODEL_ID is not None:
    df = df[df['model_id'] == ACTIVE_MODEL_ID].copy()

# Benchmark type map (VLM vs Text)
benchmarks = load_benchmarks(CONFIG_DIR / 'benchmarks.yaml')
BENCH_TYPE = {b.id: b.task_type for b in benchmarks}
df['bench_type'] = df['benchmark_id'].map(BENCH_TYPE)
print(f'Available models: {available_models}')
print(f'Active model: {ACTIVE_MODEL_ID}')
df.head()

## 1. prod (Algorithm 2) failure rates

A cell is counted as **failed** if:
- `status == 'server_crash'`, or
- `score < 0.1` (degenerate generation), or
- `n_failed / n_samples > 0.5`

In [ ]:
def cell_failed(row):
    if row['status'] in ('server_crash', 'error', 'timeout'):
        return True
    if row['score'] is None or (isinstance(row['score'], float) and np.isnan(row['score'])):
        return True
    if row['score'] < 0.1:
        return True
    if row['n_samples'] and (row['n_failed'] / max(row['n_samples'], 1)) > 0.5:
        return True
    return False

df['failed'] = df.apply(cell_failed, axis=1)

target = ['tq-3', 'tq-4', 'tq-2', 'tqp-3', 'tqp-4', 'tqp-5']
fail_rate = (
    df[df['runtime_id'].isin(target)]
    .groupby('runtime_id')['failed']
    .mean()
    .reindex(target)
)
fail_rate.to_frame('failure_rate').round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#4C9AFF' if 'tqp' not in r else '#FF6B6B' for r in fail_rate.index]
fail_rate.plot.bar(ax=ax, color=colors)
ax.set_ylabel('Cell failure rate')
ax.set_title('Algorithm 1 (MSE, blue) vs Algorithm 2 (prod/QJL, red)')
ax.set_ylim(0, 1.05)
for i, v in enumerate(fail_rate.values):
    ax.text(i, v + 0.02, f'{v:.2f}', ha='center')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 2. VLM vs Text bitwidth sensitivity

Score drop `baseline - tq-3` per benchmark type. Larger = more sensitive.

In [ ]:
pivot = df.pivot_table(
    index='runtime_id',
    columns='benchmark_id',
    values='score',
    aggfunc='mean',
)

if 'baseline' in pivot.index and 'tq-3' in pivot.index:
    drop = (pivot.loc['baseline'] - pivot.loc['tq-3']).to_frame('drop_baseline_to_tq3')
    drop['bench_type'] = drop.index.map(BENCH_TYPE)
    display(drop.round(3))

    summary = drop.groupby('bench_type')['drop_baseline_to_tq3'].agg(['mean', 'median', 'max'])
    print('\nPer benchmark type:')
    display(summary.round(3))

    fig, ax = plt.subplots(figsize=(7, 4))
    sns.boxplot(data=drop.reset_index(), x='bench_type', y='drop_baseline_to_tq3', ax=ax)
    sns.stripplot(data=drop.reset_index(), x='bench_type', y='drop_baseline_to_tq3',
                  color='black', size=6, ax=ax)
    ax.set_title('Score drop baseline -> tq-3 by benchmark type')
    ax.axhline(0, color='gray', lw=1)
    plt.tight_layout()
    plt.show()
else:
    print('baseline or tq-3 missing from results')

## 3. Asymmetric K/V effectiveness

- `tq-K4V2` (K=4,V=2, avg 3) vs `tq-3` (K=V=3)
- `tq-K4V3` (K=4,V=3, avg 3.5) vs `tq-3h` (K=V=3.5)

In [ ]:
def compare_pair(sym, asym):
    if sym not in pivot.index or asym not in pivot.index:
        print(f'missing: {sym} or {asym}')
        return None
    cmp = pd.DataFrame({
        sym: pivot.loc[sym],
        asym: pivot.loc[asym],
    })
    cmp['delta (asym - sym)'] = cmp[asym] - cmp[sym]
    cmp['bench_type'] = cmp.index.map(BENCH_TYPE)
    return cmp.round(3)

print('### avg 3-bit: tq-3 vs tq-K4V2')
display(compare_pair('tq-3', 'tq-K4V2'))

print('### avg 3.5-bit: tq-3h vs tq-K4V3')
display(compare_pair('tq-3h', 'tq-K4V3'))

In [ ]:
# Aggregate asymmetric gain
def agg_delta(sym, asym):
    cmp = compare_pair(sym, asym)
    if cmp is None:
        return None
    return cmp.groupby('bench_type')['delta (asym - sym)'].mean()

rows_ = []
for sym, asym, label in [
    ('tq-3', 'tq-K4V2', 'avg 3-bit'),
    ('tq-3h', 'tq-K4V3', 'avg 3.5-bit'),
]:
    d = agg_delta(sym, asym)
    if d is not None:
        for bt, val in d.items():
            rows_.append({'budget': label, 'bench_type': bt, 'mean_delta': val})

if rows_:
    gains = pd.DataFrame(rows_)
    display(gains.round(3))
    fig, ax = plt.subplots(figsize=(7, 4))
    sns.barplot(data=gains, x='budget', y='mean_delta', hue='bench_type', ax=ax)
    ax.axhline(0, color='gray', lw=1)
    ax.set_title('Asymmetric K/V mean score gain over symmetric')
    plt.tight_layout()
    plt.show()